# DMA U1–U2 Homework Starter

This notebook is shared by candidates A/B/C. It provides the controlled experiment, not a winning method. Change **one primary intervention** only.

Before running an intervention, submit the locked 200–300 word bet slip. Official numbers must use the published seeds and 2,048 evaluation samples.

In [ ]:
# Download the shared module when the notebook is opened in Colab.
from pathlib import Path
import urllib.request

MODULE = Path('dma_hw_starter.py')
MODULE_URL = ('https://shihhsinwang0214.github.io/personal_website/notes/notebooks/'
              'diffusion-models-applications/dma-homework/dma_hw_starter.py')
if not MODULE.exists():
    urllib.request.urlretrieve(MODULE_URL, MODULE)

import dma_hw_starter as hw
import matplotlib.pyplot as plt
import numpy as np
import torch
print('torch:', torch.__version__, '| device:', 'cuda' if torch.cuda.is_available() else 'cpu')

## 0. Identity and case

Use the same student identifier for every run. The hash is stable; it is not a random hyperparameter.

In [ ]:
STUDENT_ID = 'REPLACE_WITH_YOUR_STUDENT_ID'
ASSIGNMENT = 'B'  # 'A', 'B', or 'C'

case = hw.make_case(STUDENT_ID)
reference = hw.print_case_report(case)
hw.plot_case(case);


## 1. Controlled baseline

Use `DEV_STEPS` only to check that code runs. Every submitted table must use the official budget. If the instructor supplies a checkpoint, set `BASELINE_CHECKPOINT`; otherwise the deterministic cell trains the baseline once.

In [ ]:
from pathlib import Path

BASELINE_CHECKPOINT = None  # e.g. Path('baseline-CASEID.pt')
DEV_STEPS = None            # e.g. 200 while debugging; None for official run

if ASSIGNMENT == 'A':
    official_steps, checkpoints = 6000, (1500, 3000, 6000)
    base_config = hw.ExperimentConfig(assignment='A', path='vp',
        target_kind='eps', max_steps=official_steps, checkpoint_steps=checkpoints)
elif ASSIGNMENT == 'B':
    official_steps, checkpoints = 6000, (1500, 3000, 6000)
    base_config = hw.ExperimentConfig(assignment=ASSIGNMENT, path='linear',
        target_kind='velocity', max_steps=official_steps, checkpoint_steps=checkpoints)
else:
    official_steps, checkpoints = 3000, (750, 1500, 3000)
    base_config = hw.ExperimentConfig(assignment='C', path='vp',
        target_kind='eps', max_steps=official_steps, checkpoint_steps=checkpoints)

if DEV_STEPS is not None:
    base_config.max_steps = int(DEV_STEPS)
    base_config.checkpoint_steps = (int(DEV_STEPS),)

if BASELINE_CHECKPOINT and Path(BASELINE_CHECKPOINT).exists():
    baseline_model, base_config = hw.load_baseline(case, BASELINE_CHECKPOINT)
    baseline_run = None
else:
    baseline_run = hw.train(case, base_config)
    baseline_model = baseline_run.model
    print('baseline training:', baseline_run.ledger.summary())

In [ ]:
MAIN_NFE = {'A': 6, 'B': 8, 'C': 16}[ASSIGNMENT]
def run_sampler(model, config, nfe, *, times=None, bend_fn=hw.student_bend, eta=0.0):
    if config.path == 'vp' and config.target_kind in {'eps', 'x1', 'v'}:
        return hw.ddim_sample(model, config, nfe, eta=eta, n_samples=hw.N_EVAL, times=times)
    return hw.sample(model, config, nfe, n_samples=hw.N_EVAL, bend_fn=bend_fn, times=times)

baseline_samples, baseline_traj, baseline_sample_budget = run_sampler(
    baseline_model, base_config, MAIN_NFE)
baseline_metric = hw.evaluate(baseline_samples, case)
target = hw.forty_percent_target(baseline_metric, reference)
print('baseline:', baseline_metric.summary())
print('40% gap-closure targets:', target)
print('sampling:', baseline_sample_budget.summary())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
hw.plot_samples(baseline_samples, case, f'baseline, {MAIN_NFE} NFE', axes[0])
hw.plot_trajectories(baseline_traj, f'baseline trajectories, {MAIN_NFE} NFE', axes[1])
plt.tight_layout()

if ASSIGNMENT == 'A':
    baseline_64_samples, baseline_64_traj, _ = run_sampler(baseline_model, base_config, 64)
    baseline_64_metric = hw.evaluate(baseline_64_samples, case)
    print('baseline, 64 NFE:', baseline_64_metric.summary())
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    hw.plot_samples(baseline_64_samples, case, 'baseline, 64 NFE', axes[0])
    hw.plot_trajectories(baseline_64_traj, 'baseline trajectories, 64 NFE', axes[1])
    plt.tight_layout()

In [ ]:
baseline_bins = hw.binned_prediction_error(baseline_model, case, base_config)
hw.plot_time_error(baseline_bins, 'baseline: where is the model wrong?');


## 2. Locked bet slip — paste the submitted version

**Symptom:**  
**Causal hypothesis:**  
**One primary intervention:**  
**Predicted precision / coverage direction:**  
**Expected cost:**  

Do not rewrite this after seeing the result. Append a dated revision note instead.

## 3. One intervention

Choose only the hook allowed by your assignment. Leave all unused hooks equal to their baseline.

In [ ]:
# B/path hook: return g_phi(x0, x1) in x_t=(1-t)x0+t*x1+t(1-t)g_phi.
def my_bend(x0, x1):
    return torch.zeros_like(x0)  # TODO only if B/path is your primary intervention

# B/pairing hook: return a permutation of x1. Never change either marginal.
def my_pairing(x0, x1):
    return x1  # TODO only if B/pairing is your primary intervention

# A/time-grid hook. Keep endpoints and strict monotonicity.
def my_time_grid(nfe, path, device):
    return hw.student_time_grid(nfe, path, device)  # TODO only for this intervention

method_config = hw.ExperimentConfig(**vars(base_config))
USE_CUSTOM_TIME_GRID = False
METHOD_ETA = 0.0  # A only; changing eta is itself the one primary intervention

# Set exactly one controlled difference below, then explain it in math. Examples:
# method_config.path = 'student'; method_config.target_kind = 'velocity'  # one path intervention
# method_config.pairing = 'student'
# method_config.time_sampling = 'middle'
# method_config.weighting = 'early'
# method_config.target_kind = 'v'  # assignment C only

print('baseline config:', vars(base_config))
print('method config:  ', vars(method_config))

In [ ]:
method_run = hw.train(case, method_config, bend_fn=my_bend, pairing_fn=my_pairing)
method_model = method_run.model
print('method training:', method_run.ledger.summary())

# For an A/time-grid intervention, pass times=my_time_grid(...). Otherwise leave None.
method_path = hw.get_path(method_config.path, my_bend)
custom_times = (my_time_grid(MAIN_NFE, method_path, method_config.device)
                if USE_CUSTOM_TIME_GRID else None)

method_samples, method_traj, method_sample_budget = run_sampler(
    method_model, method_config, MAIN_NFE, bend_fn=my_bend,
    times=custom_times, eta=METHOD_ETA)
method_metric = hw.evaluate(method_samples, case)
print('method:', method_metric.summary())
print('gap closed:', hw.gap_closed(method_metric, baseline_metric, reference))
print('sampling:', method_sample_budget.summary())

In [ ]:
# Required NFE curve. Candidate C still reports the curve, but its main point is NFE=16.
nfe_values = [2, 4, 6, 12, 32] if ASSIGNMENT == 'A' else [2, 4, 8, 16, 32]
rows = []
for nfe in nfe_values:
    b_samples, _, _ = run_sampler(baseline_model, base_config, nfe)
    m_times = (my_time_grid(nfe, method_path, method_config.device)
               if USE_CUSTOM_TIME_GRID else None)
    m_samples, _, _ = run_sampler(method_model, method_config, nfe,
                                  bend_fn=my_bend, times=m_times, eta=METHOD_ETA)
    b_metric, m_metric = hw.evaluate(b_samples, case), hw.evaluate(m_samples, case)
    rows.append((nfe, b_metric.precision, b_metric.coverage, m_metric.precision, m_metric.coverage))
print('NFE | base P | base C | method P | method C')
for row in rows:
    print('%3d | %.3f | %.3f | %.3f | %.3f' % row)

## 4. Required result panels

Add the main comparison figure, common-coordinate time-bin error, and the in-run checkpoint sweep here. Candidate B must also draw at least 16 conditional paths and report path length or transport cost.

In [ ]:
# Required training-length sweep: these are checkpoints from one run, not retraining.
checkpoint_rows = []
for step in sorted(method_run.checkpoints):
    checkpoint_model = hw.model_at_checkpoint(method_run, step)
    cp_samples, _, _ = run_sampler(checkpoint_model, method_config, MAIN_NFE,
                                  bend_fn=my_bend, eta=METHOD_ETA)
    cp_metric = hw.evaluate(cp_samples, case)
    checkpoint_rows.append((step, cp_metric.precision, cp_metric.coverage))
print('step | precision | coverage')
for step, precision, coverage in checkpoint_rows:
    print(f'{step:4d} | {precision:.3f} | {coverage:.3f}')

if ASSIGNMENT == 'B':
    path_for_plot = hw.get_path(method_config.path, my_bend)
    conditional = hw.conditional_paths(
        case, path_for_plot, n_pairs=16,
        pairing_fn=my_pairing if method_config.pairing == 'student' else hw.student_pairing)
    fig, ax = plt.subplots(figsize=(5, 5))
    values = conditional.numpy()
    for j in range(values.shape[1]):
        ax.plot(values[:, j, 0], values[:, j, 1], alpha=.65)
    ax.set(aspect='equal', title='16 conditional paths')
    print('transport cost:', hw.mean_transport_cost(conditional))
    print('sampled trajectory length:', hw.mean_path_length(method_traj))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
hw.plot_samples(baseline_samples, case, 'baseline', axes[0, 0])
hw.plot_samples(method_samples, case, 'one intervention', axes[0, 1])
hw.plot_trajectories(baseline_traj, 'baseline trajectories', axes[1, 0])
hw.plot_trajectories(method_traj, 'method trajectories', axes[1, 1])
plt.tight_layout()

method_bins = hw.binned_prediction_error(
    method_model, case, method_config, bend_fn=my_bend, pairing_fn=my_pairing)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(baseline_bins['t'], baseline_bins['mse'], 'o-', label='baseline')
ax.plot(method_bins['t'], method_bins['mse'], 'o-', label='method')
ax.set(xlabel='t', ylabel='common-coordinate velocity MSE', title='Where did the intervention help?')
ax.grid(alpha=.25); ax.legend();

## 5. Your limitation micro-toy

Change one geometric condition—not merely the random seed—and make your intervention fail. Use width 32 and at most 1,000 optimizer steps. State the failure condition before running it, show the negative result, and distinguish a method limitation from unfinished optimization.

In [ ]:
# The budget guard enforces the micro-toy limits. Build the smallest case that tests
# your claimed limitation; do not perform a broad hyperparameter sweep.
micro_config = hw.ExperimentConfig(assignment=ASSIGNMENT, path=method_config.path,
    target_kind=method_config.target_kind, max_steps=1000, checkpoint_steps=(1000,),
    width=32, batch_size=128, micro_toy=True)
# TODO: define and run the minimal counterexample.

## 6. U3 interface row and revision log

| source | path | pairing | target / weighting | sampler | NFE | cost |
|---|---|---|---|---|---:|---|
| TODO | TODO | TODO | TODO | TODO | TODO | TODO |

**Revision log after seeing results:**  
**One sentence conclusion supported by the figure:**  
**One sentence stating where the conclusion does not apply:**